### Data parsing

In [1]:
from langchain_core.documents import Document 

In [2]:
doc=Document(
    page_content="you gotta check yourself before you wreck yourself",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"damru",
        "date_created":"2026-01-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'damru', 'date_created': '2026-01-01'}, page_content='you gotta check yourself before you wreck yourself')

In [3]:
import os
os.makedirs("../data/files",exist_ok=True)

In [4]:
text={
    "../data/files/intro.txt":"""I have worked as a Software engineer for 2 years in India and 3 years in the US and during this time span I’ve realized that promotions, raises, and strong performance ratings are rarely about working the hardest. They’re about creating visible, meaningful IMPACT however small or big your organization is.

I have worked at a startUp in India and now I work at Walmart, and based on my experience I have compiled principles that have consistently worked for me. None of this is company-specific. These are patterns that apply almost everywhere.

1. Integrate AI Into Your Day-to-Day Work
Every company today is looking for employees who can use AI to save time, reduce manual effort, and increase team efficiency. These are also the people who tend to be most valuable and are safe during lay-offs.

As an employee, start by identifying where your team spends the most time:

Deployments

Production releases

Manual validations

Debugging

Documentation

Repetitive analysis

Then ask a simple question:
“What part of this can be automated or accelerated using AI?”

Start small. Even if the first solution only helps you, that’s fine. Over time, turn it into something that benefits the team. The moment your work starts saving other people time, you create leverage.

Eg: Initially, you can just create an agent.md file and integrate it with cursor or copilot, for it to work like a chatbot. You can integrate this chatbot to places where product can use it to make certain repetitive and simple changes, where engineering involvement is not necessary. This will help you save engineering bandwidth.

One honest truth: You will likely need to use your weekends or personal time early on to experiment and build these things. But believe me it will be WORTH IT.

2. Choose the Right Business Initiatives
During sprint planning or roadmap discussions, I stopped asking:
“What task should I pick up?”

And started asking: “Which initiative will make the biggest impact?”

Always evaluate work from an impact perspective, not an input perspective. Promotions and raises are not tied to how busy you were they’re tied to what changed because of your work.

If you’re early in your career, you may not be given ownership of major initiatives immediately. So initially you can:

Contribute beyond your assigned scope

Support high-impact projects

Just try to be involved in these projects in some way. The closer your work is to business priorities, the easier it becomes to justify your growth during reviews.

3. Become the Point of Contact (POC)
Growth in any company requires cross-functional visibility. This means interacting not just with other developers, but also with:

Product managers

Stakeholders

Sister teams and their leads

Data scientists

Support teams, etc.

When you become the POC for an application or system, people know:

What you’re responsible for

What problems you solve

Whom to reach out to

Year-end reviews often include feedback from these people, and if you are the POC for them, their feedback automatically will be in favor of your growth!

4. Communicate Your Impact Clearly
Early on in my career, I have made this mistake of assuming that my work will speak for itself. But it DOES NOT! Hence, in order to create visibility, I’ve made it a habit to track:

Problems I solved

Risks I prevented

Metrics I moved

Cross-team contributions

When I have 1-1 meetings with my managers or leads, I clearly state the impact: I focus less on what I did and more on what changed: Reduced failures, Improved performance, Saved time or cost, Increased adoption or reliability.

I also make it a point to write everything down well in advance of the meeting and discuss each point clearly. This helps keep the conversation focused and meaningful, rather than turning into a generic discussion without clear direction.

Your manager needs concrete examples to represent you well for promotions, and you explicitly repeating your impact, helps him/her remember it and use it to present your case.

"""
}

In [5]:
for f,c in text.items():
    with open(f,'w',encoding="utf-8") as f:
        f.write(c)
print("done")

done


In [9]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader
loader=DirectoryLoader("../data/files",
glob="**/*.txt",
loader_cls=TextLoader,
loader_kwargs={'encoding':'utf-8'},
show_progress=True
)
docs=loader.load()
print(docs)

100%|██████████| 1/1 [00:00<00:00, 34.39it/s]

[Document(metadata={'source': '..\\data\\files\\intro.txt'}, page_content='I have worked as a Software engineer for 2 years in India and 3 years in the US and during this time span I’ve realized that promotions, raises, and strong performance ratings are rarely about working the hardest. They’re about creating visible, meaningful IMPACT however small or big your organization is.\n\nI have worked at a startUp in India and now I work at Walmart, and based on my experience I have compiled principles that have consistently worked for me. None of this is company-specific. These are patterns that apply almost everywhere.\n\n1. Integrate AI Into Your Day-to-Day Work\nEvery company today is looking for employees who can use AI to save time, reduce manual effort, and increase team efficiency. These are also the people who tend to be most valuable and are safe during lay-offs.\n\nAs an employee, start by identifying where your team spends the most time:\n\nDeployments\n\nProduction releases\n\nM

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader
loader=DirectoryLoader("../data/pdf",
glob="**/*.pdf",
loader_cls= PyMuPDFLoader,
show_progress=True
)
docs=loader.load()
print(docs)

100%|██████████| 1/1 [00:00<00:00,  1.75it/s]

[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'file_path': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'Untitled document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Importance of SAFERTOS (Safety-Critical \nReal-Time OS) \nSAFERTOS is a pre-certified, safety-critical RTOS designed for systems where determinism, \nreliability, and certification matter (automotive, medical devices, industrial safety).\u200b\n It is derived from FreeRTOS, but rewritten by safety experts and certified to IEC 61508 SIL3, \nISO 26262 ASIL D, IEC 62304 Class C, and more. \n1) SAFERTOS for Automotive and Supported Processors \nWhy SAFERTOS Is Used in Automotive Systems\u200b\nAutomotive systems must meet extreme safety and timing guarantees. SAFERTOS 

### Chunking

In [20]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")
all_pdf_documents

Found 1 PDF files to process

Processing: Untitled document (4) (1).pdf
  ✓ Loaded 5 pages

Total documents loaded: 5


[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}, page_content='Importance  of  SAFERTOS  (Safety-Critical  \nReal-Time\n \nOS)\n \nSAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  \nreliability,\n \nand\n \ncertification\n \nmatter\n \n(automotive,\n \nmedical\n \ndevices,\n \nindustrial\n \nsafety).\n \n \nIt\n \nis\n \nderived\n \nfrom\n \nFreeRTOS,\n \nbut\n \nrewritten\n \nby\n \nsafety\n \nexperts\n \nand\n \ncertified\n \nto\n \nIEC\n \n61508\n \nSIL3,\n \nISO\n \n26262\n \nASIL\n \nD,\n \nIEC\n \n62304\n \nClass\n \nC,\n \nand\n \nmore.\n \n1)  SAFERTOS  for  Automotive  and  Supported  Processors  \nWhy  SAFERTOS  Is  Used  in  Automotive  Systems  \nAutomotive\n \n

In [21]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks=split_documents(all_pdf_documents)
chunks

Split 5 documents into 6 chunks

Example chunk:
Content: Importance  of  SAFERTOS  (Safety-Critical  
Real-Time
 
OS)
 
SAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  
reliability,
 
and
 
certification
...
Metadata: {'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}, page_content='Importance  of  SAFERTOS  (Safety-Critical  \nReal-Time\n \nOS)\n \nSAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  \nreliability,\n \nand\n \ncertification\n \nmatter\n \n(automotive,\n \nmedical\n \ndevices,\n \nindustrial\n \nsafety).\n \n \nIt\n \nis\n \nderived\n \nfrom\n \nFreeRTOS,\n \nbut\n \nrewritten\n \nby\n \nsafety\n \nexperts\n \nand\n \ncertified\n \nto\n \nIEC\n \n61508\n \nSIL3,\n \nISO\n \n26262\n \nASIL\n \nD,\n \nIEC\n \n62304\n \nClass\n \nC,\n \nand\n \nmore.\n \n1)  SAFERTOS  for  Automotive  and  Supported  Processors  \nWhy  SAFERTOS  Is  Used  in  Automotive  Systems  \nAutomotive\n \n

### Embeddings and vector storage

In [11]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [12]:
class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully")
            print(f'Embedding dimension: {self.model.get_sentence_embedding_dimension()}')
        except Exception as e:
            raise Exception(f"Failed to load model: {e}")

    def generate_embeddings(self,texts: List[str]) ->np.ndarray:
        if not self.model:
            raise Exception("Model not loaded")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated {len(embeddings)} embeddings")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager 

c:\Users\ddali\Data science . ML . AI\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ddali\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1761.59it/s]
BertModel LOA

Model sentence-transformers/all-MiniLM-L6-v2 loaded successfully
Embedding dimension: 384


### Vector search and similarity

In [13]:
class VectorStore:
    def __init__(self,collection_name: str="pdf_documents",persist_directory: str="./data/chroma_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,metadata={"description": "cosine similarity"}
            )
            print("Vector store initialized successfully")
            print(f"Collection: {self.collection_name}")
            print(f'collection count: {self.collection.count()}')
        except Exception as e:
            print(f"Error initializing store: {e}")
            raise
    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        print(f"adding {len(documents)} documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list = []
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["source"]=doc.metadata.get("source","unknown")
            metadata["page"]=doc.metadata.get("page",0)
            metadata["chunk_index"]=i
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                documents=documents_text,
                metadatas=metadatas
            )
            print(f"Successfully added {len(documents)} documents to vector store")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise
vector_store = VectorStore()
vector_store 




Vector store initialized successfully
Collection: pdf_documents
collection count: 0


In [23]:
chunks

[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}, page_content='Importance  of  SAFERTOS  (Safety-Critical  \nReal-Time\n \nOS)\n \nSAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  \nreliability,\n \nand\n \ncertification\n \nmatter\n \n(automotive,\n \nmedical\n \ndevices,\n \nindustrial\n \nsafety).\n \n \nIt\n \nis\n \nderived\n \nfrom\n \nFreeRTOS,\n \nbut\n \nrewritten\n \nby\n \nsafety\n \nexperts\n \nand\n \ncertified\n \nto\n \nIEC\n \n61508\n \nSIL3,\n \nISO\n \n26262\n \nASIL\n \nD,\n \nIEC\n \n62304\n \nClass\n \nC,\n \nand\n \nmore.\n \n1)  SAFERTOS  for  Automotive  and  Supported  Processors  \nWhy  SAFERTOS  Is  Used  in  Automotive  Systems  \nAutomotive\n \n

In [25]:
embeddings=embedding_manager.generate_embeddings([doc.page_content for doc in chunks])
vector_store.add_documents(chunks,embeddings)


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]

Generated 6 embeddings
adding 6 documents to vector store
Successfully added 6 documents to vector store


### retreival testing 

In [28]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rr=RAGRetriever(vector_store,embedding_manager)
rr

In [31]:
rr.retrieve("rtos")

Retrieving documents for query: 'rtos'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.70it/s]

Generated 1 embeddings
Retrieved 0 documents (after filtering)


[]

## Augmentation